#### Step 1 — Project paths


In [17]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
SNOWPARK_DIR = PROJECT_ROOT / "Snowpark"

MODEL_PATH = SNOWPARK_DIR / "src" / "models" / "temperature_model.joblib"
TEST_DATA_PATH = SNOWPARK_DIR / "data" / "test_data.csv"

if str(SNOWPARK_DIR) not in sys.path:
    sys.path.append(str(SNOWPARK_DIR))


#### Step 2 — Connect to Snowflake

`get_session()` internally prints its connection dict, which includes
the password. We silence stdout for that one call so no credentials
land in the notebook output.


In [18]:
from src.utils.snowflake_connection import get_session


class _SilentWriter:
    """Discards writes — used to hide get_session()'s internal credential print."""
    def write(self, _):
        pass
    def flush(self):
        pass


_stdout_backup = sys.stdout
sys.stdout = _SilentWriter()
try:
    session = get_session("dev")
finally:
    sys.stdout = _stdout_backup

print(f"Connected: database={session.get_current_database()}, schema={session.get_current_schema()}")


Connected: database="DB_LPDG_RGMCET_MASTERCLASS", schema="SCH_LPDG_RGMCET_MASTERCLASS"


#### Step 3 — Load the trained model


In [19]:
import joblib

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Trained model not found at {MODEL_PATH}. "
        "Run Preprocessing._Training.ipynb first."
    )

trained_model = joblib.load(MODEL_PATH)
trained_model


#### Step 4 — Feature schema and test data

In [20]:
import pandas as pd

FEATURE_COLUMNS = [
    "TEMPERATURE_LAG_1", "TEMPERATURE_LAG_2", "TEMPERATURE_LAG_3", "HOUR",
    "HUMIDITY_LAG_1", "PRESSURE_LAG_1", "WIND_SPEED_LAG_1",
    "PRECIPITATION_LAG_1", "CLOUD_COVER_LAG_1",
]
TARGET_COLUMN = "TEMPERATURE"
PREDICTION_COLUMN = "PREDICTED_TEMPERATURE"

if not TEST_DATA_PATH.exists():
    raise FileNotFoundError(
        f"{TEST_DATA_PATH} not found. Copy test_data.csv (saved by "
        "Preprocessing._Training.ipynb) into this notebook's folder."
    )

test_data_pdf = pd.read_csv(TEST_DATA_PATH)
test_data_df = session.create_dataframe(test_data_pdf)

print(f"Loaded {test_data_pdf.shape[0]} test rows")
test_data_df.select(*FEATURE_COLUMNS, TARGET_COLUMN).show(5)


Loaded 293 test rows
-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"TEMPERATURE_LAG_1"  |"TEMPERATURE_LAG_2"  |"TEMPERATURE_LAG_3"  |"HOUR"  |"HUMIDITY_LAG_1"  |"PRESSURE_LAG_1"  |"WIND_SPEED_LAG_1"  |"PRECIPITATION_LAG_1"  |"CLOUD_COVER_LAG_1"  |"TEMPERATURE"  |
-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|25.3                 |25.2                 |25.6                 |19      |79                |1009.9            |9.6                 |0.0                    |100                  |25.0           |
|25.0                 |25.3                 |25.2                 |20      |81                |1009.5            |9.2                 |0.0                    |100                  |25.0  

#### Step 5 — Evaluate the model


In [21]:
from snowflake.ml.modeling.metrics import mean_absolute_error, r2_score


def evaluate_model(model, test_df, target_col, prediction_col):
    """Predict on the test set and return (predictions, metrics)."""
    predictions = model.predict(test_df)

    mae = mean_absolute_error(
        df=predictions,
        y_true_col_names=target_col,
        y_pred_col_names=prediction_col,
    )
    r2 = r2_score(
        df=predictions,
        y_true_col_name=target_col,
        y_pred_col_name=prediction_col,
    )

    print(f"MAE : {mae:.2f} degrees")
    print(f"R2  : {r2:.3f}")

    # cast to plain floats so they are JSON-serializable for the registry
    return predictions, {"mae": float(mae), "r2": float(r2)}


predictions_df, evaluation_metrics = evaluate_model(
    trained_model, test_data_df, TARGET_COLUMN, PREDICTION_COLUMN
)


MAE : 0.53 degrees
R2  : 0.927


#### Step 6 — Upload the local model to a Snowflake internal stage

An internal stage is Snowflake's own file storage. Uploading the model
here — instead of registering the local `.joblib` directly — mirrors a
real deployment: the trained artifact becomes available to any session
or user with access to the stage, not just the machine that trained it.


In [22]:
STAGE_NAME = "MODEL_STAGE"

session.sql(f"CREATE STAGE IF NOT EXISTS {STAGE_NAME}").collect()

put_result = session.file.put(
    str(MODEL_PATH),
    f"@{STAGE_NAME}",
    auto_compress=False,   # keep the .joblib byte-for-byte, no .gz wrapper
    overwrite=True,
)

put_result


[PutResult(source='temperature_model.joblib', target='temperature_model.joblib', source_size=8639695, target_size=8639696, source_compression='NONE', target_compression='NONE', status='UPLOADED', message='')]

#### Step 7 — Load the model back from the stage

Downloads the file from the stage to a local temp folder and unpickles
it. This proves the round trip works — what gets registered next comes
from the stage, not directly from the original local file.


In [23]:
import tempfile

download_dir = Path(tempfile.mkdtemp())

get_result = session.file.get(f"@{STAGE_NAME}/{MODEL_PATH.name}", str(download_dir))
get_result

staged_model_path = download_dir / MODEL_PATH.name
staged_model = joblib.load(staged_model_path)
staged_model


#### Step 8 — Open the Model Registry

Binds to the database and schema the session is already connected to.


In [24]:
from snowflake.ml.registry import Registry

MODEL_NAME = "WEATHER_TEMPERATURE_MODEL"
MODEL_VERSION = "V1"

model_registry = Registry(session=session)


#### Step 9 — Register the trained model

Logs the model as a named, versioned entry. The sample input lets the
registry infer the model's input/output signature.


In [25]:
import warnings

with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    model_version = model_registry.log_model(
        staged_model,
        model_name=MODEL_NAME,
        version_name=MODEL_VERSION,
        sample_input_data=test_data_df.select(*FEATURE_COLUMNS).limit(5),
        metrics=evaluation_metrics,
    )

print(f"Registered {MODEL_NAME} / {MODEL_VERSION} (from stage) with metrics {evaluation_metrics}")


Model logged successfully.: 100%|██████████| 6/6 [00:33<00:00,  5.66s/it]                          
Registered WEATHER_TEMPERATURE_MODEL / V1 (from stage) with metrics {'mae': 0.5332320819112608, 'r2': 0.9265768739337581}


#### Step 10 — Confirm what is registered

In [26]:
model_registry.get_model(MODEL_NAME).show_versions()

,created_on,name,aliases,comment,database_name,schema_name,model_name,is_default_version,functions,metadata,user_data,model_attributes,size,environment,runnable_in,inference_services
0,2026-09-16 01:53:16.832000-07:00,V1,"[""DEFAULT"",""FIRST"",""LAST""]",None,DB_LPDG_RGMCET_MASTERCLASS,SCH_LPDG_RGMCET_MASTERCLASS,WEATHER_TEMPERATURE_MODEL,true,"[""EXPLAIN"",""PREDICT""]","{""metrics"": {""mae"": 0.5332320819112608, ""r2"": ...",{},"{""framework"":""snowml"",""task"":""TABULAR_REGRESSI...",8656228,"{""default"":{""python_version"":""3.11"",""snowflake...","[""WAREHOUSE"",""SNOWPARK_CONTAINER_SERVICES""]",[]


#### Step 11 — Load from the registry and predict


In [27]:
registered_version = model_registry.get_model(MODEL_NAME).version(MODEL_VERSION)

predictions_df = registered_version.run(
    test_data_df.select(*FEATURE_COLUMNS),
    function_name="predict",
)

predictions_df.show(5)


---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"TEMPERATURE_LAG_1"  |"TEMPERATURE_LAG_2"  |"TEMPERATURE_LAG_3"  |"HOUR"  |"HUMIDITY_LAG_1"  |"PRESSURE_LAG_1"  |"WIND_SPEED_LAG_1"  |"PRECIPITATION_LAG_1"  |"CLOUD_COVER_LAG_1"  |"PREDICTED_TEMPERATURE"  |
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|25.3                 |25.2                 |25.6                 |19      |79                |1009.9            |9.6                 |0.0                    |100                  |25.280000000000026       |
|25.0                 |25.3                 |25.2                 |20      |81                |1009.5            |9.2                 |0.0                    |100      

#### Step 12— Set the default version

The default version is what gets served when no version is named.


In [28]:
model_registry.get_model(MODEL_NAME).default = MODEL_VERSION
print(f"Default version set to {MODEL_VERSION}")


Default version set to V1


#### Step 13 — Retrain and auto-register

Re-runs the exact pipeline from Preprocessing._Training.ipynb against
the current source table, evaluates the new model, works out the next
version number, and registers it. Promotion to default stays manual so
a worse model can never take over automatically.


In [ ]:
from snowflake.snowpark import Window
from snowflake.snowpark import functions as F
from snowflake.ml.modeling.ensemble import RandomForestRegressor

SOURCE_TABLE = "TBL_WEATHER_DATA"
COLUMNS_TO_DROP = ["LATITUDE", "LONGITUDE", "TIMEZONE", "ELEVATION"]
OTHER_SENSOR_COLS = ["HUMIDITY", "PRESSURE", "WIND_SPEED", "PRECIPITATION", "CLOUD_COVER"]


def build_training_data(session, table_name, target_col, n_lags=3):
    """Clean + feature-engineer the raw table, exactly as in the training notebook."""
    df = session.table(table_name)
    df = df.select([F.col(c).alias(c.upper()) for c in df.columns])

    # Cleaning: types -> drop null targets -> dedupe -> drop station metadata -> sort
    df = df.with_column("TIMESTAMP", F.to_timestamp_ntz(F.col("TIMESTAMP")))
    df = df.filter(F.col(target_col).is_not_null()).distinct()
    df = df.drop(*[c for c in COLUMNS_TO_DROP if c in df.columns])
    df = df.sort(F.col("TIMESTAMP").asc())

    # Feature engineering: only past values, never the current hour
    time_window = Window.order_by(F.col("TIMESTAMP").asc())
    for lag in range(1, n_lags + 1):
        df = df.with_column(f"{target_col}_LAG_{lag}", F.lag(F.col(target_col), lag).over(time_window))
    for col in OTHER_SENSOR_COLS:
        df = df.with_column(f"{col}_LAG_1", F.lag(F.col(col), 1).over(time_window))
    df = df.with_column("HOUR", F.hour(F.col("TIMESTAMP")))

    # Drop warm-up rows that have no lag history yet
    condition = F.lit(True)
    for c in [c for c in df.columns if "_LAG_" in c]:
        condition = condition & F.col(c).is_not_null()
    return df.filter(condition)


def split_by_time(df, split_fraction=0.8):
    """Chronological split — never shuffled, matching the training notebook."""
    split_point = int(df.count() * split_fraction)
    numbered = df.with_column("ROW_NUM", F.row_number().over(Window.order_by(F.col("TIMESTAMP").asc())))
    train_df = numbered.filter(F.col("ROW_NUM") <= split_point).drop("ROW_NUM")
    test_df = numbered.filter(F.col("ROW_NUM") > split_point).drop("ROW_NUM")
    return train_df, test_df


def next_version_name(registry, model_name):
    """Return the next V-number, e.g. V1 -> V2. Falls back to V1 for a new model."""
    try:
        existing = registry.get_model(model_name).show_versions()["name"].tolist()
    except Exception:
        return "V1"
    numbers = [int(v[1:]) for v in existing if v.startswith("V") and v[1:].isdigit()]
    return f"V{max(numbers) + 1}" if numbers else "V1"


def retrain_and_register(session, registry, model_name):
    """Retrain on current data and register the result as the next version."""
    feature_df = build_training_data(session, SOURCE_TABLE, TARGET_COLUMN)
    train_df, test_df = split_by_time(feature_df)
    print(f"Training rows: {train_df.count()} | Test rows: {test_df.count()}")

    model = RandomForestRegressor(
        input_cols=FEATURE_COLUMNS,
        label_cols=[TARGET_COLUMN],
        output_cols=[PREDICTION_COLUMN],
        n_estimators=100,
        random_state=42,
    )
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        model.fit(train_df)
        # even after using ignore warnings, still getting the warning due to the SDK's Internal warning

        _, metrics = evaluate_model(model, test_df, TARGET_COLUMN, PREDICTION_COLUMN)

        version_name = next_version_name(registry, model_name)
        registry.log_model(
            model,
            model_name=model_name,
            version_name=version_name,
            sample_input_data=train_df.select(*FEATURE_COLUMNS).limit(5),
            metrics=metrics,
        )


    print(f"Registered {model_name} / {version_name} with metrics {metrics}")
    return version_name, metrics


new_version, new_metrics = retrain_and_register(session, model_registry, MODEL_NAME)


Training rows: 1168 | Test rows: 293


Package 'snowflake-telemetry-python' is not installed in the local environment. Your UDF might not work when the package is installed on the server but not on your local environment.


MAE : 0.53 degrees
R2  : 0.927
Model logged successfully.: 100%|██████████| 6/6 [00:32<00:00,  5.37s/it]                          
Registered WEATHER_TEMPERATURE_MODEL / V6 with metrics {'mae': 0.5332320819112607, 'r2': 0.9265768739337581}


# Testing the model

#### Step 1 — Load the model from the registry



In [33]:
registered_version = model_registry.get_model(MODEL_NAME).version(MODEL_VERSION)

print(f"Loaded {MODEL_NAME} / {MODEL_VERSION} from registry")
print("Registered metrics:", registered_version.show_metrics())


Loaded WEATHER_TEMPERATURE_MODEL / V1 from registry
Registered metrics: {'mae': 0.5332320819112608, 'r2': 0.9265768739337581}


#### Step 2 — Run predictions

Runs inference inside Snowflake on the held-out test data. The full
DataFrame is passed so TIMESTAMP and the true TEMPERATURE survive into
the results for plotting and scoring.


In [22]:
predictions_df = registered_version.run(test_data_df, function_name="predict")

predictions_df.select("TIMESTAMP", TARGET_COLUMN, PREDICTION_COLUMN).show(5)


-----------------------------------------------------------------
|"TIMESTAMP"          |"TEMPERATURE"  |"PREDICTED_TEMPERATURE"  |
-----------------------------------------------------------------
|2026-09-03 19:00:00  |25.0           |25.280000000000026       |
|2026-09-03 20:00:00  |25.0           |24.726999999999997       |
|2026-09-03 21:00:00  |24.9           |24.492000000000004       |
|2026-09-03 22:00:00  |24.6           |24.41100000000001        |
|2026-09-03 23:00:00  |24.3           |24.342999999999996       |
-----------------------------------------------------------------



#### Step 3 — Score the predictions

All four metrics are computed server-side in Snowflake, never pulled
into pandas.


In [23]:
from snowflake.ml.modeling.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score,
)

mae = mean_absolute_error(
    df=predictions_df, y_true_col_names=TARGET_COLUMN, y_pred_col_names=PREDICTION_COLUMN
)
rmse = mean_squared_error(
    df=predictions_df, y_true_col_names=TARGET_COLUMN, y_pred_col_names=PREDICTION_COLUMN,
    squared=False,   # squared=False returns RMSE instead of MSE
)
mape = mean_absolute_percentage_error(
    df=predictions_df, y_true_col_names=TARGET_COLUMN, y_pred_col_names=PREDICTION_COLUMN
)
r2 = r2_score(
    df=predictions_df, y_true_col_name=TARGET_COLUMN, y_pred_col_name=PREDICTION_COLUMN
)

test_metrics = {"mae": float(mae), "rmse": float(rmse), "mape": float(mape), "r2": float(r2)}

print(f"MAE  : {mae:.2f} °C      (average miss)")
print(f"RMSE : {rmse:.2f} °C      (punishes large misses)")
print(f"MAPE : {mape:.2%}        (average miss as a percentage)")
print(f"R2   : {r2:.3f}          (1.0 = perfect)")


MAE  : 0.53 °C      (average miss)
RMSE : 0.68 °C      (punishes large misses)
MAPE : 1.93%        (average miss as a percentage)
R2   : 0.927          (1.0 = perfect)


#### Step 4 — Bring results local for plotting

Plotly needs local data, so we pull the small result set down once and
add error columns. Scoring above already happened in Snowflake.


In [24]:
results_pdf = predictions_df.select(
    "TIMESTAMP", "HOUR", TARGET_COLUMN, PREDICTION_COLUMN
).to_pandas().sort_values("TIMESTAMP")

# Signed error shows bias (over- vs under-prediction); absolute error shows size
results_pdf["ERROR"] = results_pdf[PREDICTION_COLUMN] - results_pdf[TARGET_COLUMN]
results_pdf["ABS_ERROR"] = results_pdf["ERROR"].abs()

print(f"{len(results_pdf)} test predictions")
results_pdf.head()


293 test predictions


,TIMESTAMP,HOUR,TEMPERATURE,PREDICTED_TEMPERATURE,ERROR,ABS_ERROR
108,2026-09-03 19:00:00,19,25.0,25.280,0.280,0.280
145,2026-09-03 20:00:00,20,25.0,24.727,-0.273,0.273
182,2026-09-03 21:00:00,21,24.9,24.492,-0.408,0.408
219,2026-09-03 22:00:00,22,24.6,24.411,-0.189,0.189
256,2026-09-03 23:00:00,23,24.3,24.343,0.043,0.043


#### Step 5 — Actual vs predicted over time

The headline plot. If the two lines sit on top of each other, the model
is tracking the real temperature.


In [26]:
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=results_pdf["TIMESTAMP"], y=results_pdf[TARGET_COLUMN],
    name="Actual", line=dict(width=2),
))
fig.add_trace(go.Scatter(
    x=results_pdf["TIMESTAMP"], y=results_pdf[PREDICTION_COLUMN],
    name="Predicted", line=dict(width=2, dash="dash"),
))
fig.update_layout(
    title="Actual vs Predicted Temperature (Test Set)",
    xaxis_title="Time", yaxis_title="Temperature (°C)",
)
fig.show()


#### Step 6 — Predicted vs actual scatter

Every point is one hour. The dashed diagonal is perfect prediction —
the tighter the cloud hugs it, the better the model.


In [27]:
import plotly.express as px

axis_min = min(results_pdf[TARGET_COLUMN].min(), results_pdf[PREDICTION_COLUMN].min())
axis_max = max(results_pdf[TARGET_COLUMN].max(), results_pdf[PREDICTION_COLUMN].max())

fig = px.scatter(
    results_pdf, x=TARGET_COLUMN, y=PREDICTION_COLUMN, opacity=0.6,
    title=f"Predicted vs Actual (R² = {r2:.3f})",
)
# Reference line y = x, i.e. a perfect prediction
fig.add_shape(
    type="line", x0=axis_min, y0=axis_min, x1=axis_max, y1=axis_max,
    line=dict(dash="dash", width=2),
)
fig.update_layout(xaxis_title="Actual (°C)", yaxis_title="Predicted (°C)")
fig.show()


## Step 7 — Errors over time

Points above zero mean the model predicted too warm, below zero too
cold. A healthy model scatters randomly around zero with no drift.


In [28]:
fig = px.scatter(
    results_pdf, x="TIMESTAMP", y="ERROR", opacity=0.6,
    title="Prediction Error Over Time",
)
fig.add_hline(y=0, line_dash="dash")
fig.update_layout(xaxis_title="Time", yaxis_title="Error (Predicted − Actual, °C)")
fig.show()


#### Step 8 — Distribution of errors

Should be a narrow bell centred on zero. A shifted centre means the
model is systematically biased warm or cold.


In [29]:
fig = px.histogram(
    results_pdf, x="ERROR", nbins=30,
    title=f"Error Distribution (MAE = {mae:.2f} °C)",
)
fig.add_vline(x=0, line_dash="dash")
fig.update_layout(xaxis_title="Error (°C)", yaxis_title="Count")
fig.show()


#### Step 9 — Where the model struggles

Average error broken down by hour. Tall bars show times of day the
model finds hardest — often sunrise and sunset, when temperature moves
fastest.


In [30]:
hourly_error = (
    results_pdf.groupby("HOUR", as_index=False)["ABS_ERROR"].mean()
    .rename(columns={"ABS_ERROR": "MEAN_ABS_ERROR"})
)

fig = px.bar(
    hourly_error, x="HOUR", y="MEAN_ABS_ERROR",
    title="Average Absolute Error by Hour of Day",
)
fig.add_hline(y=mae, line_dash="dash", annotation_text="overall MAE")
fig.update_layout(xaxis_title="Hour (0–23)", yaxis_title="Mean Absolute Error (°C)")
fig.show()


#### Step 10 — Test summary

Final scorecard for the registered model version.


In [31]:
worst_hour = hourly_error.loc[hourly_error["MEAN_ABS_ERROR"].idxmax()]

print(f"Model      : {MODEL_NAME} / {MODEL_VERSION}")
print(f"Test rows  : {len(results_pdf)}")
print("-" * 40)
print(f"MAE        : {mae:.2f} °C")
print(f"RMSE       : {rmse:.2f} °C")
print(f"MAPE       : {mape:.2%}")
print(f"R²         : {r2:.3f}")
print("-" * 40)
print(f"Mean bias  : {results_pdf['ERROR'].mean():+.2f} °C "
      f"({'runs warm' if results_pdf['ERROR'].mean() > 0 else 'runs cold'})")
print(f"Worst hour : {int(worst_hour['HOUR']):02d}:00 "
      f"({worst_hour['MEAN_ABS_ERROR']:.2f} °C avg error)")


Model      : WEATHER_TEMPERATURE_MODEL / V1
Test rows  : 293
----------------------------------------
MAE        : 0.53 °C
RMSE       : 0.68 °C
MAPE       : 1.93%
R²         : 0.927
----------------------------------------
Mean bias  : -0.18 °C (runs cold)
Worst hour : 10:00 (0.86 °C avg error)


# Inference


#### Step 1 — Load the model from the registry

Loads by version, defaulting to whichever version is currently marked
default — so promoting a new version changes what serves, with no code
change here.


In [35]:
def load_serving_model(registry, model_name, version_name=None):
    """Return a registered model version — the default one unless named."""
    registered_model = registry.get_model(model_name)
    version = registered_model.version(version_name) if version_name else registered_model.default
    print(f"Serving: {model_name} / {version.version_name}")
    return version


serving_version = load_serving_model(model_registry, MODEL_NAME)


Serving: WEATHER_TEMPERATURE_MODEL / V1


#### Step 2 — Load new data

Uses test_data.csv — data the model never saw during training — and
takes the most recent rows to stand in for newly arrived readings.


In [36]:
def get_new_data(session, csv_path, n_rows=10):
    """Load unseen rows from CSV and return the most recent ones as a Snowpark DataFrame."""
    new_data_pdf = pd.read_csv(csv_path).sort_values("TIMESTAMP").tail(n_rows)
    print(f"Loaded {len(new_data_pdf)} new rows from {csv_path.name}")
    return session.create_dataframe(new_data_pdf)


new_data_df = get_new_data(session, TEST_DATA_PATH, n_rows=10)
new_data_df.select("TIMESTAMP", *FEATURE_COLUMNS).show(5)


Loaded 10 new rows from test_data.csv


c:\Users\VeeraRohithReddy\OneDrive - Lehmann & Pioneers Digital GmbH\LPDG 2026\Masterclass\LPDG_RGMCET_MASTERCLASS\venv\Lib\site-packages\snowflake\snowpark\session.py:3362: UserWarning: Pandas Dataframe has non-standard index of type <class 'pandas.core.indexes.range.RangeIndex'> which will not be written. Consider changing the index to pd.RangeIndex(start=0,...,step=1) or call reset_index() to keep index as column(s)
  success, _, _, ci_output = write_pandas(


-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"TIMESTAMP"          |"TEMPERATURE_LAG_1"  |"TEMPERATURE_LAG_2"  |"TEMPERATURE_LAG_3"  |"HOUR"  |"HUMIDITY_LAG_1"  |"PRESSURE_LAG_1"  |"WIND_SPEED_LAG_1"  |"PRECIPITATION_LAG_1"  |"CLOUD_COVER_LAG_1"  |
-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|2026-09-15 14:00:00  |28.3                 |28.5                 |29.4                 |14      |64                |1010.2            |3.6                 |0.1                    |38                   |
|2026-09-15 15:00:00  |28.1                 |28.3                 |28.5                 |15      |66                |1010.9            |1.7                 |0.0                    |29 

#### Step 3 — Run predictions

Inference runs inside Snowflake — the data never leaves the warehouse.


In [37]:
def predict(version, input_df):
    """Run the registered model on new data and return predictions."""
    return version.run(input_df, function_name="predict")


inference_df = predict(serving_version, new_data_df)
inference_df.select("TIMESTAMP", TARGET_COLUMN, PREDICTION_COLUMN).show(5)


-----------------------------------------------------------------
|"TIMESTAMP"          |"TEMPERATURE"  |"PREDICTED_TEMPERATURE"  |
-----------------------------------------------------------------
|2026-09-15 14:00:00  |28.1           |27.389000000000006       |
|2026-09-15 15:00:00  |27.8           |27.369000000000018       |
|2026-09-15 16:00:00  |27.4           |27.09200000000003        |
|2026-09-15 17:00:00  |27.0           |25.999999999999996       |
|2026-09-15 18:00:00  |26.4           |25.754                   |
-----------------------------------------------------------------



#### Step 4 — Predictions on new data

Actual values appear alongside because this data is historical. In a
live forecast the actual would not exist yet.


In [38]:
inference_pdf = (
    inference_df.select("TIMESTAMP", TARGET_COLUMN, PREDICTION_COLUMN)
    .to_pandas()
    .sort_values("TIMESTAMP")
)

inference_pdf[PREDICTION_COLUMN] = inference_pdf[PREDICTION_COLUMN].round(2)
inference_pdf["ERROR"] = (inference_pdf[PREDICTION_COLUMN] - inference_pdf[TARGET_COLUMN]).round(2)

print(f"Mean absolute error on new data: {inference_pdf['ERROR'].abs().mean():.2f} °C")
inference_pdf


Mean absolute error on new data: 0.45 °C


,TIMESTAMP,TEMPERATURE,PREDICTED_TEMPERATURE,ERROR
0,2026-09-15 14:00:00,28.1,27.39,-0.71
2,2026-09-15 15:00:00,27.8,27.37,-0.43
4,2026-09-15 16:00:00,27.4,27.09,-0.31
5,2026-09-15 17:00:00,27.0,26.00,-1.00
6,2026-09-15 18:00:00,26.4,25.75,-0.65
7,2026-09-15 19:00:00,25.9,25.85,-0.05
8,2026-09-15 20:00:00,25.4,25.73,0.33
9,2026-09-15 21:00:00,25.0,25.32,0.32
1,2026-09-15 22:00:00,24.6,24.80,0.20
3,2026-09-15 23:00:00,24.1,24.65,0.55
